# Классификация проектов с помощью линейной регрессии

In [17]:
# install libraries

%pip install numpy==1.23.5
%pip install typer==0.9.4
%pip install torch==2.0.1
%pip install transformers==4.34.0
%pip install sentence-transformers==3.0.0
%pip install spacy==3.5.4
%pip install tensorflow==2.12.0
%pip install torchtext==0.15.2
%pip install nltk==3.7
%pip install scipy==1.15.3
%pip install gensim==4.4.0
%pip install xgboost==1.7.6
%pip install catboost

%pip check

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --up

In [14]:
# library deps
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import nltk
import pandas as pd
from gensim.models.word2vec import Word2Vec
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from xgboost import XGBClassifier

In [15]:
import catboost
print(xgboost.__version__)

ModuleNotFoundError: No module named 'catboost'

## Загрузка данных

In [9]:
Labels = [
    "Автомобильные дороги",
    "Водоотведение",
    "Водопроводы",
    "Газоны дорожки",
    "Газопроводы",
    "Горные выработки",
    "Железнодорожные пути",
    "Заводы фабрики",
    "Здания",
    "Инженерное обеспечение",
    "Инфраструктура наземного электротранспорта",
    "Линии электропередачи",
    "Метрополитены",
    "Мосты и тоннели",
    "Наружное освещение",
    "Нефтепроводы",
    "Сооружения",
    "Теплопроводы",
    "Технологические установки",
]

ColumnNames = ["id", "project_name", "label"]


def load_labeled_data(path):
    labeled_dataframes = [
        pd.read_csv(f"{path}//{label}.csv", names=ColumnNames, header=0)
        for label in tqdm(Labels)
    ]
    result_df = pd.concat(labeled_dataframes)
    result_df["project_name"] = result_df["project_name"].str.strip('"')
    return result_df


def load_unlabeled_data(path):
    return pd.read_csv(
        path, sep=";", encoding="utf-8", nrows=200000, names=["id", "project_name"]
    )


# raw_df = load_unlabeled_data(f'../Data/Реестр 2022-2024 clean.csv')
# raw_df

## Разделение данных на тестовую и обучающую выборки

In [10]:
def data_train_test_split(data, labels):
    assert len(data) == len(
        labels
    ), "Размеры списков данных и результатов разметки не совпадают"
    le = LabelEncoder()
    le.fit(labels)
    y = le.transform(labels)
    return train_test_split(data, y, test_size=0.2, random_state=42)

## Способы векторизации

In [11]:
def vectorize_words_with_word2vec(sentences, vector_size):
    nltk.download("punkt")
    tokenized_sentences = [
        nltk.tokenize.word_tokenize(text.lower(), language="russian")
        for text in tqdm(sentences)
    ]
    sentence_vectors = Word2Vec(
        tokenized_sentences,
        workers=8,
        vector_size=vector_size,
        min_count=3,
        window=5,
        epochs=15,
    )
    return sentence_vectors


def vectorize_with_word2vec(sentences, vector_size):
    result = []
    for word in word_tokenize(text.lower()):
        if word in model_tweets.wv:
            result.append(model_tweets.wv[word])

    if len(result):
        result = np.average(result, axis=0)
    else:
        result = np.zeros(300)
    return result

In [12]:
# Вернет матрицу размера (len(sentences, 1024)
def vectorize_with_sentence_transformer(model_name, sentences):
    model = SentenceTransformer(model_name)
    return model.encode(sentences.to_numpy())

## Модели

In [28]:
# LogisticRegression
def log_reg(X_train, y_train):
    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)
    return model

# LogisticRegression with GridSearch
def log_reg_grid_search(X_train, y_train):
    param_grid_test = {
        "C": [1],
        "penalty": ["l1"],
        "solver": ["liblinear"],
        "max_iter": [100],
    }
    
    param_grid = {
        "C": [0.1, 1, 10],
        #"C": [0.01, 0.1, 1, 10, 100],
        "penalty": ["l1", "l2"],
        "solver": ["liblinear", "saga"],
        #"solver": ["liblinear"],
        #"max_iter": [500, 1000, 3000],
        "max_iter": [500, 1000],
    }

    grid_search = GridSearchCV(
        estimator=LogisticRegression(random_state=42),
        param_grid=param_grid,
        cv=2,
        scoring="accuracy",
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    return grid_search.best_estimator_, grid_search.best_params_

# XGBoost
def xgb(X_train, y_train):
    xgb = XGBClassifier(
        n_jobs=-1, eval_metric="logloss", tree_method="gpu_hist", random_state=42
    )
    xgb.fit(X_train, y_train)
    return xgb

# XGBoost with GridSearch
def xgb_train_grid_search(X_train, y_train):
    test_param_grid = {
        "n_estimators": [100],
        "max_depth": [6],
        "learning_rate": [0.5],
    }

    param_grid = {
        "n_estimators": [50, 80, 100, 150, 200],
        "max_depth": [4, 6, 8, 10],
        "learning_rate": [0, 1, 0.3, 0.5],
    }

    grid_search = GridSearchCV(
        estimator=XGBClassifier(
            n_jobs=-1, eval_metric="logloss", tree_method="gpu_hist", random_state=42
        ),
        param_grid=param_grid,
        scoring="accuracy",
        cv=2,
        n_jobs=-1,
    )
    grid_search.fit(X_train, y_train)
    return grid_search.best_estimator_, grid_search.best_params_

## Классификация проектов

### Загрузим данные

In [14]:
df = load_labeled_data("../Data/Reestr/Размеченные")
df

100%|██████████| 19/19 [00:00<00:00, 117.78it/s]


,id,project_name,label
0,000352ac-d728-470f-b466-dbaf03df82ad,Строительство автомобильной дороги общего поль...,Автомобильные дороги
1,0004719e-2520-4e50-92ab-b26be1e9ed0e,Капитальный ремонт автомобильной дороги по ул....,Автомобильные дороги
2,000f1480-522e-4e22-94fd-539f1e93a22b,Капитальный ремонт автомобильной дороги общего...,Автомобильные дороги
3,0012f905-46ab-4648-b519-2f1f7e2c2b1a,Капитальный ремонт автомобильной дороги Р-241 ...,Автомобильные дороги
4,001ae5dc-0cb7-40be-b52e-5e33dec04fb7,Реконструкция дорожного покрытия ул. Пограничн...,Автомобильные дороги
...,...,...,...
95,06ee6bfe-d776-474c-8209-c4d666aa947f,"Строительство блока отстойников на УППН ""Сухан...",Технологические установки
96,06f42c68-acde-49a5-9cf5-5cb1e4edd89a,Реконструкция опасного производственного объек...,Технологические установки
97,072313c2-4757-49ab-8e42-0aa2e1a3de12,"Кусты №4Б, №15, №59, №64Б Сыморьяхского местор...",Технологические установки
98,077a2af1-b8af-4ce6-bb0e-e8edde2200f4,Обустройство куста скважин № 407б Тагринского ...,Технологические установки


### Векторизуем различными способами

In [9]:
# Векторизация WordToVek
# word2vec_vectors = vectorize_with_word2vec(df['project_name'], 300)

# word2vec_vectors.wv.most_similar('мост')

In [15]:
# Векторизация с помощью SentenceTransformer с использованием модели 'sberbank-ai/sbert_large_nlu_ru'
sbert_vectors = vectorize_with_sentence_transformer(
    "sberbank-ai/sbert_large_nlu_ru", df["project_name"]
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### Разделим данные на тестовую и обучающую выборки

In [16]:
X_train, X_test, y_train, y_test = data_train_test_split(sbert_vectors, df["label"])

### Классификация 1. Логистическая регрессия + sbert sentence transformer

In [17]:
log_reg_model = log_reg(X_train, y_train)
y_pred = log_reg_model.predict(X_test)

print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Точность: 0.7526315789473684
              precision    recall  f1-score   support

           0       0.81      0.89      0.85        19
           1       0.69      0.61      0.65        18
           2       0.45      0.45      0.45        22
           3       0.91      0.83      0.87        24
           4       0.83      0.65      0.73        23
           5       0.65      0.68      0.67        25
           6       1.00      0.88      0.94        17
           7       0.74      0.88      0.80        16
           8       0.56      0.82      0.67        11
           9       0.65      0.87      0.74        23
          10       0.69      1.00      0.81        11
          11       0.76      0.80      0.78        20
          12       0.90      0.95      0.93        20
          13       0.94      0.77      0.85        22
          14       1.00      0.84      0.91        25
          15       0.82      0.78      0.80        18
          16       0.86      0.23      0.36        2

In [29]:
log_reg_model_gs, log_reg_model_gs_params = log_reg_grid_search(X_train, y_train)
y_pred = log_reg_model_gs.predict(X_test)

print("Точность:", accuracy_score(y_test, y_pred))
print("Лучшие параметры:", log_reg_model_gs_params)
print(classification_report(y_test, y_pred))

Точность: 0.8315789473684211
Лучшие параметры: {'C': 10, 'max_iter': 500, 'penalty': 'l2', 'solver': 'saga'}
              precision    recall  f1-score   support

           0       0.81      0.89      0.85        19
           1       0.81      0.94      0.87        18
           2       0.75      0.68      0.71        22
           3       0.95      0.83      0.89        24
           4       1.00      0.78      0.88        23
           5       0.95      0.76      0.84        25
           6       0.94      1.00      0.97        17
           7       0.81      0.81      0.81        16
           8       0.69      0.82      0.75        11
           9       0.77      0.87      0.82        23
          10       0.85      1.00      0.92        11
          11       0.83      0.75      0.79        20
          12       0.76      0.95      0.84        20
          13       0.94      0.77      0.85        22
          14       1.00      1.00      1.00        25
          15       0.79   

### Вариант классификации 2. XGBoost + sbert sentence transformer

In [46]:
xgb_model = xgb(X_train, y_train)
y_pred = xgb_model.predict(X_test)

print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Точность: 0.8078947368421052
              precision    recall  f1-score   support

           0       0.64      0.84      0.73        19
           1       0.78      0.78      0.78        18
           2       0.58      0.64      0.61        22
           3       0.95      0.75      0.84        24
           4       0.95      0.78      0.86        23
           5       0.83      0.80      0.82        25
           6       0.94      0.94      0.94        17
           7       0.73      0.69      0.71        16
           8       0.77      0.91      0.83        11
           9       0.72      0.91      0.81        23
          10       1.00      1.00      1.00        11
          11       0.88      0.70      0.78        20
          12       0.79      0.95      0.86        20
          13       0.86      0.86      0.86        22
          14       0.92      0.96      0.94        25
          15       0.71      0.83      0.77        18
          16       0.87      0.50      0.63        2

### Вариант классификации. XGBoost + sbert sentence transformer + оптимизация гипперпараметров с GridSearch

In [54]:
xgb_model_gs, xgb_model_gs_params = xgb_train_grid_search(X_train, y_train)
y_pred = xgb_model_gs.predict(X_test)

print("Точность:", accuracy_score(y_test, y_pred))
print("Лучшие параметры:", xgb_model_gs_params)
print(classification_report(y_test, y_pred))

Точность: 0.8184210526315789
              precision    recall  f1-score   support

           0       0.74      0.89      0.81        19
           1       0.80      0.67      0.73        18
           2       0.62      0.59      0.60        22
           3       0.95      0.83      0.89        24
           4       0.90      0.83      0.86        23
           5       0.88      0.88      0.88        25
           6       0.94      0.94      0.94        17
           7       0.79      0.69      0.73        16
           8       0.83      0.91      0.87        11
           9       0.67      0.87      0.75        23
          10       0.85      1.00      0.92        11
          11       0.93      0.70      0.80        20
          12       0.83      0.95      0.88        20
          13       0.86      0.86      0.86        22
          14       0.89      1.00      0.94        25
          15       0.71      0.83      0.77        18
          16       0.88      0.54      0.67        2